In [6]:
%pip install -q curl_cffi cryptography

import re
import json
import zlib
import base64
import time
from curl_cffi import requests as cffi_requests
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# ── PrivateBin helpers ────────────────────────────────────────────────────────
_BASE58 = '123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz'

def _base58_decode(s):
    num = 0
    for ch in s:
        num = num * 58 + _BASE58.index(ch)
    result = b''
    while num > 0:
        num, rem = divmod(num, 256)
        result = bytes([rem]) + result
    return b'\x00' * (len(s) - len(s.lstrip('1'))) + result

def decrypt_privatebin(url, password=''):
    """Decrypt a PrivateBin v2 paste without a browser.
    The decryption key lives in the URL fragment and never touches the server.
    """
    paste_url, url_key = url.split('#', 1)
    paste_id = paste_url.split('?')[-1]
    base_url  = paste_url[:paste_url.index('?')]

    res  = cffi_requests.get(f"{base_url}?{paste_id}",
                             headers={"Accept": "application/json"},
                             impersonate="chrome",
                             timeout=60)
    data = res.json()
    if data.get('status') != 0:
        raise ValueError(f"PrivateBin API error: {data}")

    adata = data['adata']
    iv_b64, salt_b64, iterations, key_size, _tag_size, _algo, _mode, compression = adata[0]

    iv   = base64.b64decode(iv_b64)
    salt = base64.b64decode(salt_b64)
    ct   = base64.b64decode(data['ct'])

    key_input = _base58_decode(url_key)
    if password:
        key_input += password.encode()

    derived = PBKDF2HMAC(hashes.SHA256(), key_size // 8, salt, iterations).derive(key_input)
    aad     = json.dumps(adata, separators=(',', ':')).encode()

    raw = AESGCM(derived).decrypt(iv, ct, aad)

    if compression == 'zlib':
        raw = zlib.decompress(raw, wbits=-15)   # raw deflate

    # The paste payload is {"paste": "...markdown..."}
    payload = json.loads(raw.decode())
    return payload.get('paste', raw.decode())
# ─────────────────────────────────────────────────────────────────────────────

def extract_download_links_from_pages(urls, max_retries=3):
    # fuckingfast.co now uses HTMX: the download button POSTs to /f/{id}/go
    # and the actual CDN download URL is returned in the `hx-redirect` response header.
    extracted_links = []
    failed_urls = list(urls)

    for attempt in range(max_retries):
        if not failed_urls:
            break
        if attempt > 0:
            print(f"\n--- Retry attempt {attempt}/{max_retries-1} for {len(failed_urls)} failed URLs ---")
            time.sleep(3)

        still_failing = []
        for url in failed_urls:
            try:
                # Strip the hash fragment and extract the file ID from the path
                file_id = url.split('#')[0].rstrip('/').split('/')[-1]
                page_url = f"https://fuckingfast.co/{file_id}"

                htmx_headers = {
                    "Referer": page_url,
                    "HX-Request": "true",
                    "HX-Current-URL": page_url,
                    "HX-Target": "null",
                }

                response = cffi_requests.post(
                    f"https://fuckingfast.co/f/{file_id}/go",
                    headers=htmx_headers,
                    impersonate="chrome"
                )

                download_link = response.headers.get("hx-redirect")

                if download_link:
                    extracted_links.append(download_link)
                else:
                    print(f"  No hx-redirect for {url} | status={response.status_code} | body={response.text[:120]}")
                    still_failing.append(url)

            except Exception as e:
                print(f"  Error for {url}: {type(e).__name__}: {e}")
                still_failing.append(url)

        failed_urls = still_failing

    if failed_urls:
        print(f"\n\n{'='*50}")
        print(f"FAILED after {max_retries} attempts ({len(failed_urls)} URLs):")
        print('='*50)
        for url in failed_urls:
            print(url)

    return extracted_links

link_inputs = '''
# TODOWN

# https://fitgirl-repacks.site/assassins-creed-black-flag-resynced/
# Assassin’s Creed: Black Flag Resynced – Deluxe Edition, v1.0.6 + 10 DLCs/Bonuses
https://paste.fitgirl-repacks.site/?b7f1f779cba7bfbc#GupC3qoJmtxSCbKUriCn5WoMMhVbExvAzLS4Y4Y5M79U




# https://fuckingfast.co/i2f26tc5q47l#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part01.rar
# https://fuckingfast.co/hr6ojn4wrv7i#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part02.rar
# https://fuckingfast.co/ypsl9c56b8dr#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part03.rar
# https://fuckingfast.co/0gw5ccbjr3cj#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part04.rar
# https://fuckingfast.co/x96353167o4y#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part05.rar
# https://fuckingfast.co/rfynvbyay46p#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part06.rar
# https://fuckingfast.co/w6sz5it03hfs#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part07.rar
# https://fuckingfast.co/d0xgkahg22lj#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part08.rar
# https://fuckingfast.co/jhfnyu4cd2lb#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part09.rar
# https://fuckingfast.co/1nt3g2sf2s1x#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part10.rar
# https://fuckingfast.co/uyqf3yfp6nu6#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part11.rar
# https://fuckingfast.co/f2yein0af0x3#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part12.rar
# https://fuckingfast.co/xqsj71kdbvtr#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part13.rar
# https://fuckingfast.co/2zwbjduiwvsj#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part14.rar
# https://fuckingfast.co/htnbcown9n1a#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part15.rar
# https://fuckingfast.co/bjb54t7zcvgf#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part16.rar
# https://fuckingfast.co/yuvwbyd65btw#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part17.rar
# https://fuckingfast.co/h37yz8leosux#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part18.rar
# https://fuckingfast.co/f4hjeq66ml2b#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part19.rar
# https://fuckingfast.co/z2k7xnbqss80#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part20.rar
# https://fuckingfast.co/f4re4egkwjv9#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part21.rar
# https://fuckingfast.co/7xft39n78qo7#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part22.rar
# https://fuckingfast.co/js9ywdkqjklk#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part23.rar
# https://fuckingfast.co/48tee3rb30e6#The_Talos_Principle_Reawakened_--_fitgirl-repacks.site_--_.part24.rar

# https://fuckingfast.co/t53azsl6j81y#The.Talos.Principle.Reawakened.Update.v1.01b-RUNE.rar

# https://fuckingfast.co/u5h1dbxkcwqr#fg-optional-videos.part01.rar
# https://fuckingfast.co/ldhutgkee2nq#fg-optional-videos.part02.rar
# https://fuckingfast.co/h8q3jxitony0#fg-optional-videos.part03.rar
# https://fuckingfast.co/v9h70g18omp8#fg-optional-videos.part04.rar
# https://fuckingfast.co/utub5e4mvetl#fg-optional-videos.part05.rar
# https://fuckingfast.co/ob50bevleofd#fg-optional-videos.part06.rar
# https://fuckingfast.co/g9ll6fmfu2uk#fg-optional-videos.part07.rar
# https://fuckingfast.co/z1535p8dsxfk#fg-optional-videos.part08.rar
# https://fuckingfast.co/7i5l1xxjbr47#fg-optional-videos.part09.rar
# https://fuckingfast.co/9samp27y7sv3#fg-optional-videos.part10.rar
# https://fuckingfast.co/fnxxwl4ut97a#fg-optional-videos.part11.rar
# https://fuckingfast.co/qa5b1n7prmot#fg-optional-videos.part12.rar
# https://fuckingfast.co/kpcrvuizodpe#fg-optional-videos.part13.rar








# https://fitgirl-repacks.site/escape-simulator-2/
# Escape Simulator 2 – v16494r + Bonus OST + Online Co-op
# https://paste.fitgirl-repacks.site/?7cccc488979584a1#3Wf4HjWYNynRbPr3pRiJK6KdmngWoE5GsV45roGkctma

# https://fuckingfast.co/qh0zlqvwd6p5#Escape.Simulator.2.Update.v20448r-RUNE.part4.rar
# https://fuckingfast.co/abbq99wkjv70#Escape.Simulator.2.Update.v20448r-RUNE.part3.rar
# https://fuckingfast.co/rio1wdccl83k#Escape.Simulator.2.Update.v20448r-RUNE.part2.rar
# https://fuckingfast.co/j2fpdr7274as#Escape.Simulator.2.Update.v20448r-RUNE.part1.rar
# https://fuckingfast.co/i9umifidu89j#Escape.Simulator.2.Update.v18159r-RUNE.rar
# https://fuckingfast.co/ymlauzpwdfch#Escape.Simulator.2.Update.v18158r-RUNE.part3.rar
# https://fuckingfast.co/x95vq0jk48x7#Escape.Simulator.2.Update.v18158r-RUNE.part2.rar
# https://fuckingfast.co/5747nqh6abej#Escape.Simulator.2.Update.v18158r-RUNE.part1.rar

# https://fuckingfast.co/p08epchlftdu#STALKER_2_Heart_of_Chornobyl_Update_from_v1.8.1_to_v1.9.0-ElAmigos.part3.rar
# https://fuckingfast.co/22wwmrpz5h24#STALKER_2_Heart_of_Chornobyl_Update_from_v1.8.1_to_v1.9.0-ElAmigos.part2.rar
# https://fuckingfast.co/juim3a8tf1lu#STALKER_2_Heart_of_Chornobyl_Update_from_v1.8.1_to_v1.9.0-ElAmigos.part1.rar
# https://fuckingfast.co/3j0b0qv12va9#S.T.A.L.K.E.R.2.Heart.of.Chornobyl.Update.v1.8.1-RUNE.part3.rar
# https://fuckingfast.co/bctz22weer4m#S.T.A.L.K.E.R.2.Heart.of.Chornobyl.Update.v1.8.1-RUNE.part2.rar
# https://fuckingfast.co/149xc53iukcb#S.T.A.L.K.E.R.2.Heart.of.Chornobyl.Update.v1.8.1-RUNE.part1.rar
# https://fuckingfast.co/w07njcxws0sn#STALKER_2_Heart_of_Chornobyl_Update_from_v1.6.1_to_v1.7.0-ElAmigos.part10.rar
# https://fuckingfast.co/t7cpn73zjwoq#STALKER_2_Heart_of_Chornobyl_Update_from_v1.6.1_to_v1.7.0-ElAmigos.part09.rar
# https://fuckingfast.co/8hdwuprtdwmr#STALKER_2_Heart_of_Chornobyl_Update_from_v1.6.1_to_v1.7.0-ElAmigos.part08.rar
# https://fuckingfast.co/evy3hpj7xj1e#STALKER_2_Heart_of_Chornobyl_Update_from_v1.6.1_to_v1.7.0-ElAmigos.part07.rar
# https://fuckingfast.co/9v5806qr11ms#STALKER_2_Heart_of_Chornobyl_Update_from_v1.6.1_to_v1.7.0-ElAmigos.part06.rar
# https://fuckingfast.co/2rvo2yq9lhf5#STALKER_2_Heart_of_Chornobyl_Update_from_v1.6.1_to_v1.7.0-ElAmigos.part05.rar
# https://fuckingfast.co/li7v0gj8bsmt#STALKER_2_Heart_of_Chornobyl_Update_from_v1.6.1_to_v1.7.0-ElAmigos.part04.rar
# https://fuckingfast.co/y3zjo9ma9tv6#STALKER_2_Heart_of_Chornobyl_Update_from_v1.6.1_to_v1.7.0-ElAmigos.part03.rar
# https://fuckingfast.co/cpr31q1b61hx#STALKER_2_Heart_of_Chornobyl_Update_from_v1.6.1_to_v1.7.0-ElAmigos.part02.rar
# https://fuckingfast.co/o9yrxn1mr841#STALKER_2_Heart_of_Chornobyl_Update_from_v1.6.1_to_v1.7.0-ElAmigos.part01.rar





# Marvel’s Spider-Man 2: Digital Deluxe Edition, v1.130.1.0/v1.131.0.0 + 2 DLCs + Unlocker + Bonus Soundtrack

# https://fitgirl-repacks.site/marvels-spider-man-2/
# https://paste.fitgirl-repacks.site/?aa7ee62c43546e4f#HdgTUYUKFGRt2uVMyfx4i2KaZ8eS31oLoUJwzMaZqKJ7
# https://fuckingfast.co/l2fk9bbr2kyu#Marvels_SpiderMan_2_Update_from_v1.526_to_v2.629-ElAmigos.rar
# https://fuckingfast.co/jehskk36zfs5#Marvels.Spider-Man.2.Update.v1.526.0.0-RUNE.rar
# https://fuckingfast.co/j5udcsk6284z#Marvels.Spider-Man.2.Update.v1.318.1.0-RUNE.part2.rar
# https://fuckingfast.co/ojpf10i53kno#Marvels.Spider-Man.2.Update.v1.318.1.0-RUNE.part1.rar


# Palworld – v1.0.0.100427 (Release) + Bonus OST
# https://fitgirl-repacks.site/palworld/
# https://paste.fitgirl-repacks.site/?7eae4d3ee5df85dc#8kAMZ5rfPhxkb47vbUDY49zHuk4b6BNmfK7fVH1qfoDG
# https://fuckingfast.co/3k4omhw8o0xj#Palworld.Build.24088745.Steamworks.Fix-SOVEREIGN.rar

# The Last Gas Station – v1.0.0.304 + Bonus OST
# https://fuckingfast.co/5xiomf6lm7kh#The_Last_Gas_Station_--_fitgirl-repacks.site_--_.rar

# Dying Light: The Beast Restored Land – Definitive Edition, v1.6.0 + 11 DLCs/Bonuses + Multiplayer
# https://fitgirl-repacks.site/dying-light-the-beast/
# https://paste.fitgirl-repacks.site/?086392109b40ecd4#AWqomRjN17h25ATe6hqetLGQqUEDeLTdz9kCNjAKbjeG
# https://fuckingfast.co/e77sfvg834ad#Dying_Light_The_Beast_Update_from_v1.6.0_to_v1.6.2-ElAmigos.rar
# https://fuckingfast.co/5dfvkreznao6#Dying_Light_The_Beast_Update_from_v1.6.2_to_v1.6.3-ElAmigos.rar


'''


links = [x.strip() for x in link_inputs.splitlines() if x.strip() and not x.strip().startswith('#')]

for link_input in links:
  urls_to_process = []

  if link_input.startswith('https://paste.fitgirl'):
    try:
      markdown = decrypt_privatebin(link_input)
      urls_to_process = re.findall(r'https://fuckingfast\.co/\S+', markdown)
    except cffi_requests.errors.Timeout as e:
      print(f"  Timeout error while decrypting PrivateBin link {link_input}: {e}. Skipping.")
      continue
    except Exception as e:
      print(f"  Error decrypting PrivateBin link {link_input}: {e}. Skipping.")
      continue

  elif link_input.startswith('https://fuckingfast.co'):
    urls_to_process = link_input.split(' ')
    # print('fuckingfast')
  else:
    print("Unsupported link type. Please provide a paste.fitgirl or fuckingfast.co link.")

  if urls_to_process:
      all_download_links = extract_download_links_from_pages(urls_to_process)

      if all_download_links:
          # print("\nAll Extracted Download Links:")
          for download_link in all_download_links:
              print(download_link+'\n')
              # print('\n')
      else:
          print("\nNo download links were extracted from the provided URLs.")
  else:
      print("\nNo URLs to process.")

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-45' coro=<_extract_all() done, defined at /tmp/ipykernel_577/3961112515.py:130> exception=Exception('Browser.close: Connection closed while reading from the driver')>
Traceback (most recent call last):
  File "/usr/lib/python3.13/asyncio/tasks.py", line 304, in __step_run_and_handle_result
    result = coro.send(None)
  File "/tmp/ipykernel_577/3961112515.py", line 178, in _extract_all
    await browser.close()
  File "/usr/local/lib/python3.13/dist-packages/playwright/async_api/_generated.py", line 16458, in close
    return mapping.from_maybe_impl(await self._impl_obj.close(reason=reason))
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/playwright/_impl/_browser.py", line 244, in close
    raise e
  File "/usr/local/lib/python3.13/dist-packages/playwright/_impl/_browser.py", line 241, in close
    await self._channel.send("clo

  No hx-redirect for https://fuckingfast.co/yjoj5ds9m14e#ACBF_Resynced_CRACKED_--_fitgirl-repacks.site_--_.part01.rar | status=403 | body=captcha verification failed
  No hx-redirect for https://fuckingfast.co/nis0jv8ckvje#ACBF_Resynced_CRACKED_--_fitgirl-repacks.site_--_.part02.rar | status=403 | body=captcha verification failed
  No hx-redirect for https://fuckingfast.co/n38fzkl0805c#ACBF_Resynced_CRACKED_--_fitgirl-repacks.site_--_.part03.rar | status=403 | body=captcha verification failed
  No hx-redirect for https://fuckingfast.co/7y8hyav2l6o2#ACBF_Resynced_CRACKED_--_fitgirl-repacks.site_--_.part04.rar | status=403 | body=captcha verification failed
  No hx-redirect for https://fuckingfast.co/5xw8s181jkce#ACBF_Resynced_CRACKED_--_fitgirl-repacks.site_--_.part05.rar | status=403 | body=captcha verification failed
  No hx-redirect for https://fuckingfast.co/p0z5pqbhajgu#ACBF_Resynced_CRACKED_--_fitgirl-repacks.site_--_.part06.rar | status=403 | body=captcha verification failed
  No

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  OPTION C — manual Turnstile solve INSIDE Colab (free, no proxy / no solver)
#  Cell 1/3: install a virtual display + VNC + noVNC web viewer + Playwright.
#  Run this once per session (takes ~1-2 min).
# ═══════════════════════════════════════════════════════════════════════════════
%pip install -q playwright pyvirtualdisplay nest_asyncio
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y xvfb x11vnc novnc websockify fluxbox > /dev/null 2>&1
!playwright install --with-deps chromium
print("✓ install done")


Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Cell 2/3: start display + window manager + VNC, then launch a VISIBLE browser
#  and embed the live view. If the frame is blank, use the "open in new tab" link.
# ═══════════════════════════════════════════════════════════════════════════════
import os, time, subprocess, shutil, asyncio, nest_asyncio
from pyvirtualdisplay import Display
from playwright.async_api import async_playwright
from google.colab.output import serve_kernel_port_as_iframe, serve_kernel_port_as_window

nest_asyncio.apply()
loop = asyncio.get_event_loop()

# ── diagnostics: are the binaries present? ────────────────────────────────────
for b in ("Xvfb", "x11vnc", "websockify", "fluxbox"):
    print(f"{b:12} -> {shutil.which(b)}")
NOVNC = next((d for d in ["/usr/share/novnc", "/usr/share/webapps/novnc",
                          "/usr/lib/novnc"] if os.path.isdir(d)), None)
HTML = "vnc.html" if NOVNC and os.path.exists(f"{NOVNC}/vnc.html") else "vnc_lite.html"
print("novnc dir  ->", NOVNC, "| client:", HTML)

# ── 1) fresh virtual X display ────────────────────────────────────────────────
try:
    _DISPLAY.stop()
except Exception:
    pass
_DISPLAY = Display(visible=0, size=(1360, 768))
_DISPLAY.start()
DISP = os.environ["DISPLAY"]
print("DISPLAY    ->", DISP)

# ── 2) window manager (without it the browser window never maps → blank view) ─
subprocess.run(["pkill", "-f", "fluxbox"], stderr=subprocess.DEVNULL)
subprocess.Popen(["fluxbox"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# ── 3) VNC server + noVNC web client, with logs so we can see failures ────────
subprocess.run(["pkill", "-f", "x11vnc"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "websockify"], stderr=subprocess.DEVNULL)
xp = subprocess.Popen(["x11vnc", "-display", DISP, "-forever", "-shared",
                       "-nopw", "-rfbport", "5900"],
                      stdout=open("/tmp/x11vnc.log", "w"), stderr=subprocess.STDOUT)
wp = subprocess.Popen(["websockify", "--web", NOVNC or "/usr/share/novnc",
                       "6080", "localhost:5900"],
                      stdout=open("/tmp/wsk.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)
print("x11vnc     -> pid", xp.pid, "alive:", xp.poll() is None)
print("websockify -> pid", wp.pid, "alive:", wp.poll() is None)

# ── 4) launch a persistent VISIBLE browser now (kept in globals for Cell 3) ───
_pw = loop.run_until_complete(async_playwright().start())
_browser = loop.run_until_complete(_pw.chromium.launch(
    headless=False,
    args=["--no-sandbox", "--disable-dev-shm-usage",
          "--disable-blink-features=AutomationControlled",
          "--window-position=0,0", "--window-size=1360,768"]))
_ctx = loop.run_until_complete(_browser.new_context(
    viewport={"width": 1360, "height": 740},
    user_agent=("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36")))
_page = loop.run_until_complete(_ctx.new_page())
loop.run_until_complete(_page.goto("https://fuckingfast.co/i2f26tc5q47l",
                                   wait_until="domcontentloaded"))
print("browser    -> launched & navigated to a test page")

# ── 5) show it ────────────────────────────────────────────────────────────────
qs = f"/{HTML}?autoconnect=true&resize=scale&reconnect=true"
serve_kernel_port_as_iframe(6080, path=qs, height=820)
print("\nIf the frame above is blank, click the button below to open it in a tab:")
serve_kernel_port_as_window(6080, path=qs)


DISPLAY = :0


<IPython.core.display.Javascript object>

↑ live browser view. Run Cell 3, then solve the Turnstile here and click DOWNLOAD.


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Cell 3/3: drive the browser opened in Cell 2 and capture the CDN links.
#  For each URL: it loads in the view above → solve the Turnstile if shown, then
#  click DOWNLOAD (once for ads, again to start). The hx-redirect is grabbed
#  automatically. After the first solve the rest usually auto-clear.
# ═══════════════════════════════════════════════════════════════════════════════
FF_URLS = [
    "https://fuckingfast.co/i2f26tc5q47l#The_Talos_Principle_Reawakened_--_.part01.rar",
    # add more fuckingfast.co links here
]

WAIT_SECONDS_PER_URL = 300  # time allowed for you to solve each page

async def grab_links(urls, wait_s=WAIT_SECONDS_PER_URL):
    results = []
    for url in urls:
        fid = url.split('#')[0].rstrip('/').split('/')[-1]
        got = {}

        async def on_resp(r, fid=fid, got=got):
            if r.request.method == "POST" and r.url.rstrip('/').endswith(f"/f/{fid}/go"):
                loc = await r.header_value("hx-redirect")
                if loc:
                    got["link"] = loc

        _page.on("response", on_resp)
        await _page.goto(f"https://fuckingfast.co/{fid}", wait_until="domcontentloaded")
        print(f"\n>>> {fid}: solve Turnstile in the view above, then click DOWNLOAD.")

        for _ in range(wait_s * 2):
            if "link" in got:
                break
            await asyncio.sleep(0.5)

        _page.remove_listener("response", on_resp)
        if "link" in got:
            print("   ✔", got["link"])
            results.append(got["link"])
        else:
            print("   ✗ timed out for", fid)
    return results

_links = loop.run_until_complete(grab_links(FF_URLS))
print("\n================ RESULTS ================")
for _l in _links:
    print(_l)



>>> i2f26tc5q47l: solve Turnstile in the view above, then click DOWNLOAD.
   ✗ timed out for i2f26tc5q47l

================ RESULTS ================


### link will be lik either

```https://filekeeper.net/5y9fgeiiawb2/ACBF_Resynced_CRACKED_--_fitgirl-repacks.site_--_.part01.rar```

or 

```https://paste.fitgirl-repacks.site/?9b83a394a1d920a1#HNJUSConLNWoHU8vdkuAaHPzjQuAe8C3tShNzbWNTCT9
```

but here is the way it work 

you open this 
https://filekeeper.net/5y9fgeiiawb2/ACBF_Resynced_CRACKED_--_fitgirl-repacks.site_--_.part01.rar

it goes to this 

https://filekeeper.net/download

you wait 5 sec it will appear like 

[text](HTMLS/filekeeper.html)

you get link like that 

https://tunnel1.dlproxy.uk/download/j974CvvJb3EFHa_LkHxUOdqg9cltPioZ1XaNcnrwKcdHPye7wX0w7hD_Ko0Y2z77-lSIW5n0w2SygpXbsYvnGX0gfznaZP5sgLoVK6MGje1ZD5qyvmRKT1mp0DBLzfSv3WRTNlk8yF2p6cQX31pLcCcQMu9BbPeSUU2t3R1QHtmzIv3adr0SajqAk7G0j6m09apAxoeJ8ecav1Q1br-jRoRRYMMlqYPQlTpEciThbTbQ72U8LvV_6mGze7MU9pZi5wdsDiWb0kW5kMki9TwJxm3-37cITcOYskOT56Qh-U1C0L7EnKf1s3WG3VW8QctLHtvpY5_3vmYbTwB0nvXuTwgcUa1VBtBXdB8ZPGrWjwiysBYPkWj7AFmzHXqlYcY9QxKHDXxnQ0fiaa4va6QLuDMDd9EgiLZOoGdRZ2vOuToBJvs1erP02Q74O1-kzJDq69FBvrLMTjVFb_wgCc6x-S3kL_TnsNKBoiYxOnJs6ClHTDsOae1Zr2Of_5kr1Gy1aQxFeWEqOFurSd0-KJLorYPscKDAoP9qpQvHmyeZBe1qbdcKXlV0UA_MbAuP5Q_NfypmDMNbNcj8GgKplAg1z8lkZr8vP2t8FPVLNBB_Y7oxon8ZKkEylTbjR9TJZN0C-VJsfpXy0e4mIbMosKX3mw?sig=KwzhiOIca0sHqJUQA2SmP4sYukcKbsW0DT0h78N6L5c	




In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
#  filekeeper.net extractor — NO browser, NO captcha solver.
#  filekeeper is a standard XFileSharing site: the download page runs a countdown
#  then POSTs `op=download2` back to the file URL; the CDN link (dlproxy.uk) comes
#  back in the 302 Location header. We just replicate that POST with curl_cffi.
#  Accepts filekeeper.net links directly, or a paste.fitgirl link that decrypts
#  to filekeeper links. Self-contained: run this cell alone, no other cell needed.
# ═══════════════════════════════════════════════════════════════════════════════
%pip install -q curl_cffi cryptography

import re
import json
import zlib
import time
import base64
from curl_cffi import requests as cffi_requests
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# ── PrivateBin decrypt (paste.fitgirl links) — no browser ─────────────────────
_BASE58 = '123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz'


def _base58_decode(s):
    num = 0
    for ch in s:
        num = num * 58 + _BASE58.index(ch)
    result = b''
    while num > 0:
        num, rem = divmod(num, 256)
        result = bytes([rem]) + result
    return b'\x00' * (len(s) - len(s.lstrip('1'))) + result


def decrypt_privatebin(url, password=''):
    """Decrypt a PrivateBin v2 paste. The key lives in the URL fragment."""
    paste_url, url_key = url.split('#', 1)
    paste_id = paste_url.split('?')[-1]
    base_url = paste_url[:paste_url.index('?')]

    res = cffi_requests.get(f"{base_url}?{paste_id}",
                            headers={"Accept": "application/json"},
                            impersonate="chrome", timeout=60)
    data = res.json()
    if data.get('status') != 0:
        raise ValueError(f"PrivateBin API error: {data}")

    adata = data['adata']
    iv_b64, salt_b64, iterations, key_size, _tag, _algo, _mode, compression = adata[0]
    iv = base64.b64decode(iv_b64)
    salt = base64.b64decode(salt_b64)
    ct = base64.b64decode(data['ct'])

    key_input = _base58_decode(url_key)
    if password:
        key_input += password.encode()

    derived = PBKDF2HMAC(hashes.SHA256(), key_size // 8, salt, iterations).derive(key_input)
    aad = json.dumps(adata, separators=(',', ':')).encode()
    raw = AESGCM(derived).decrypt(iv, ct, aad)
    if compression == 'zlib':
        raw = zlib.decompress(raw, wbits=-15)
    payload = json.loads(raw.decode())
    return payload.get('paste', raw.decode())


def extract_filekeeper_link(url, session=None, wait_countdown=True):
    """Return the direct CDN link for one filekeeper.net file page (or None)."""
    url = url.split('#')[0].strip()
    s = session or cffi_requests.Session(impersonate="chrome")

    page = s.get(url, timeout=60)
    if page.status_code != 200:
        print(f"  GET {page.status_code} for {url}")
        return None

    def _attr(name, default=""):
        m = re.search(rf'data-{name}="([^"]*)"', page.text)
        return m.group(1) if m else default

    code = _attr("code") or url.rstrip('/').split('/')[-1]
    if _attr("has-captcha") == "true":
        print(f"  {code}: page requires a captcha — needs the browser flow, skipping.")
        return None

    if wait_countdown:
        try:
            time.sleep(int(_attr("countdown", "5")) + 1)
        except ValueError:
            time.sleep(6)

    data = {
        "op": "download2",
        "id": code,
        "rand": _attr("rand"),
        "referer": _attr("referer"),
        "method_free": _attr("method") or "Free download",
        "down_direct": "1",
    }
    resp = s.post(url, data=data, timeout=60, allow_redirects=False)
    link = resp.headers.get("location")
    if not link:
        m = re.search(r'https://[^\s"\'<>]+dlproxy[^\s"\'<>]+', resp.text)
        link = m.group(0) if m else None
    if not link:
        print(f"  {code}: no CDN link (status={resp.status_code}).")
    return link


def extract_filekeeper_links(link_inputs):
    """Process a multi-line block of filekeeper.net and/or paste.fitgirl links."""
    lines = [x.strip() for x in link_inputs.splitlines()
             if x.strip() and not x.strip().startswith('#')]

    results = []
    session = cffi_requests.Session(impersonate="chrome")
    for line in lines:
        if line.startswith("https://paste.fitgirl"):
            try:
                markdown = decrypt_privatebin(line)
                targets = re.findall(r'https://filekeeper\.net/\S+', markdown)
            except Exception as e:
                print(f"  Error decrypting {line}: {e}. Skipping.")
                continue
        elif "filekeeper.net" in line:
            targets = [line]
        else:
            print(f"  Unsupported link: {line}")
            continue

        for t in targets:
            link = extract_filekeeper_link(t, session=session)
            if link:
                print(f"  \u2713 {link}")
                results.append(link)
    return results


FILEKEEPER_INPUTS = '''
# One link per line. Lines starting with # are ignored.
# Paste filekeeper.net links or a paste.fitgirl-repacks.site link.

https://paste.fitgirl-repacks.site/?9b83a394a1d920a1#HNJUSConLNWoHU8vdkuAaHPzjQuAe8C3tShNzbWNTCT9
'''

_fk_links = extract_filekeeper_links(FILEKEEPER_INPUTS)
print(f"\n================ RESULTS ({len(_fk_links)}) ================")
for _i, _l in enumerate(_fk_links, 1):
    print(f"{_i:>2}. {_l}")

if _fk_links:
    with open("filekeeper_links.txt", "w", encoding="utf-8") as _f:
        _f.write("\n".join(_fk_links))
    print(f"\nSaved {len(_fk_links)} link(s) to filekeeper_links.txt")


  ✓ https://tunnel1.dlproxy.uk/download/j974CvvJb3EFHa_LkHxUOdqg9cltPioZ1XaNcnrwKcdHPye7wX0w7hD_Ko0Y2z77-lSIW5n0w2SygpXbsYvnGX0gfznaZP5sgLoVK6MGje1ZD5qyvmRKT1mp0DBLzfSv3WRTNlk8yF2p6cQX31pLcCcQMu9BbPeSUU2t3R1QHtmzIv3adr0SajqAk7G0j6m09apAxoeJ8ecav1Q1br-jRoRRYMMlqYPQlTpEciThbTbQ72U8LvV_6mGze7MU9pZi5wdsDiWb0kW5kMki9TwJxm3-37cITcOYskOT56Qh-U1C0L7EnKf1s3WG3VW8QctLHtvpY5_3vmYbTwB0nvXuTwgcUa1VBtBXdB8ZPGrWjwiysBYPkWj7AFmzHXqlYcY9QxKHDXxnQ0fiaa4va6QLuDMDd9EgiLZOoGdRZ2vOuToBJvs1erP02Q74O1-kzJDq69FBvrLMTjVFb_wgCc6x-S3kL_TnsNKBoiYxOnJs6CmYDGMl6afmAsstjmUF9L68yUvrb1UYJHzUam6WaO1e_WkXKVPC9Lz3fwn2ZH8wNhIQlMR7jOn1f2ktCULaqwOSO78KoGY-MiSUEno-outMxyoyDB4fEpzqJvUDUc3_Rsh-jIlql16hzhSrplTxRIBHAmazCyDpg5_fUYJHBu1rxg?sig=4PyaXApdadM7lQQYI-awJO5X6oenDHdpkOglu3n6MFk
  ✓ https://tunnel1.dlproxy.uk/download/j974CvvJb3EFHa_LkHxUOdqg9cltPioZ1XaNcnrwKcdHPye7wX0w7hD_Ko0Y2z77-lSIW5n0w2SygpXbsYvnGX0gfznaZP5sgLoVK6MGje0tl0dnrCuF7qykp1fCx3boal9Cv4ZpBbEotYjqXZoqH_DbTnRVrFpsxMvZrmxaA2pLttRJUCNWL7TOR1XN4IDSjNOPGpGIk1zTCEGd5

In [ ]:
%pip install -q google-colab-selenium webdriver-manager undetected-chromedriver

import google_colab_selenium as gs
import undetected_chromedriver as uc

from IPython.core.display import display, HTML
from datetime import date
from datetime import datetime
from selenium import webdriver
from selenium.webdriver import Chrome
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
